# 07 Daily Watchlist Demo

Generate and display a daily hitter watchlist.



In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import save_csv

print(PROJECT_ROOT)

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting


In [2]:
pred_path = PROJECT_ROOT / "data/predictions/model_predictions_2025_04_01_to_05_31.csv"

pred_df = pd.read_csv(pred_path)
pred_df["prediction_date"] = pd.to_datetime(pred_df["prediction_date"])

print(pred_df.shape)
pred_df.head()

(6125, 9)


,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm
0,455117,2025-05-11,0.148583,0.314537,0.440423,0.388523,0.312607,0.292462,0.276183
1,456781,2025-05-11,0.419369,0.206825,0.204182,0.224426,0.294077,0.293146,0.347680
2,457705,2025-05-11,0.274772,0.366160,0.399297,0.413562,0.327504,0.322416,0.327371
3,457759,2025-05-11,0.443777,0.286520,0.261328,0.282639,0.301951,0.308217,0.322703
4,467793,2025-05-11,0.322851,0.288303,0.337477,0.322596,0.322525,0.284948,0.282033


In [3]:
matchup_path = PROJECT_ROOT / "data/features/matchup_features_2025_04_01_to_05_31.csv"

matchup_df = pd.read_csv(matchup_path)

print(matchup_df.shape)
matchup_df[[
    "batter",
    "pitcher",
    "matchup_PA",
    "matchup_xwOBA",
    "matchup_weakness_score",
    "matchup_label"
]].head()

(34205, 44)


,batter,pitcher,matchup_PA,matchup_xwOBA,matchup_weakness_score,matchup_label
0,455117,543294,2,0.0000,0.040450,Unfavorable
1,455117,592332,1,0.1470,0.002978,Neutral
2,455117,592791,2,0.0565,0.092852,Unfavorable
3,455117,594902,2,0.0045,0.064052,Unfavorable
4,455117,596133,1,0.2400,0.008405,Neutral


In [4]:
batter_matchup_summary = (
    matchup_df
    .groupby("batter")
    .agg(
        avg_matchup_weakness_score=("matchup_weakness_score", "mean"),
        median_matchup_weakness_score=("matchup_weakness_score", "median"),
        n_matchups=("pitcher", "count"),
        avg_matchup_PA=("matchup_PA", "mean"),
        historical_matchup_xwOBA=("matchup_xwOBA", "mean"),
    )
    .reset_index()
)

batter_matchup_summary.head()

,batter,avg_matchup_weakness_score,median_matchup_weakness_score,n_matchups,avg_matchup_PA,historical_matchup_xwOBA
0,455117,0.056475,0.051748,41,1.731707,0.285034
1,456781,0.061840,0.063358,48,1.520833,0.260680
2,457705,-0.021942,-0.021409,98,1.918367,0.351806
3,457759,0.005244,0.010843,55,1.654545,0.311287
4,467793,0.001919,0.007082,110,1.854545,0.321326


In [5]:
label_counts = (
    matchup_df
    .pivot_table(
        index="batter",
        columns="matchup_label",
        values="pitcher",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

label_counts.columns.name = None

for col in ["Favorable", "Neutral", "Unfavorable"]:
    if col not in label_counts.columns:
        label_counts[col] = 0

label_counts["total_labeled_matchups"] = (
    label_counts["Favorable"] + label_counts["Neutral"] + label_counts["Unfavorable"]
)

label_counts["favorable_matchup_rate"] = (
    label_counts["Favorable"] / label_counts["total_labeled_matchups"]
)

label_counts["unfavorable_matchup_rate"] = (
    label_counts["Unfavorable"] / label_counts["total_labeled_matchups"]
)

label_counts.head()

,batter,Favorable,Neutral,Unfavorable,total_labeled_matchups,favorable_matchup_rate,unfavorable_matchup_rate
0,455117,0,14,27,41,0.000000,0.658537
1,456781,0,9,39,48,0.000000,0.812500
2,457705,35,61,2,98,0.357143,0.020408
3,457759,7,40,8,55,0.127273,0.145455
4,467793,11,93,6,110,0.100000,0.054545


In [6]:
batter_matchup_summary = batter_matchup_summary.merge(
    label_counts[
        [
            "batter",
            "favorable_matchup_rate",
            "unfavorable_matchup_rate",
            "total_labeled_matchups",
        ]
    ],
    on="batter",
    how="left"
)

batter_matchup_summary.head()

,batter,avg_matchup_weakness_score,median_matchup_weakness_score,n_matchups,avg_matchup_PA,historical_matchup_xwOBA,favorable_matchup_rate,unfavorable_matchup_rate,total_labeled_matchups
0,455117,0.056475,0.051748,41,1.731707,0.285034,0.000000,0.658537,41
1,456781,0.061840,0.063358,48,1.520833,0.260680,0.000000,0.812500,48
2,457705,-0.021942,-0.021409,98,1.918367,0.351806,0.357143,0.020408,98
3,457759,0.005244,0.010843,55,1.654545,0.311287,0.127273,0.145455,55
4,467793,0.001919,0.007082,110,1.854545,0.321326,0.100000,0.054545,110


In [7]:
pred_df["prediction_date"].value_counts().sort_index().head()

prediction_date
2025-05-11    339
2025-05-12    340
2025-05-13    334
2025-05-14    322
2025-05-15    329
Name: count, dtype: int64

In [8]:
pred_df["prediction_date"].value_counts().sort_index().head()

prediction_date
2025-05-11    339
2025-05-12    340
2025-05-13    334
2025-05-14    322
2025-05-15    329
Name: count, dtype: int64

In [9]:
demo_date = pd.Timestamp("2025-05-11")

In [10]:
watchlist = pred_df[pred_df["prediction_date"] == demo_date].copy()

print(watchlist.shape)
watchlist.head()

(339, 9)


,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm
0,455117,2025-05-11,0.148583,0.314537,0.440423,0.388523,0.312607,0.292462,0.276183
1,456781,2025-05-11,0.419369,0.206825,0.204182,0.224426,0.294077,0.293146,0.347680
2,457705,2025-05-11,0.274772,0.366160,0.399297,0.413562,0.327504,0.322416,0.327371
3,457759,2025-05-11,0.443777,0.286520,0.261328,0.282639,0.301951,0.308217,0.322703
4,467793,2025-05-11,0.322851,0.288303,0.337477,0.322596,0.322525,0.284948,0.282033


In [11]:
watchlist = watchlist.merge(
    batter_matchup_summary,
    on="batter",
    how="left"
)

watchlist.head()

,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm,avg_matchup_weakness_score,median_matchup_weakness_score,n_matchups,avg_matchup_PA,historical_matchup_xwOBA,favorable_matchup_rate,unfavorable_matchup_rate,total_labeled_matchups
0,455117,2025-05-11,0.148583,0.314537,0.440423,0.388523,0.312607,0.292462,0.276183,0.056475,0.051748,41,1.731707,0.285034,0.000000,0.658537,41
1,456781,2025-05-11,0.419369,0.206825,0.204182,0.224426,0.294077,0.293146,0.347680,0.061840,0.063358,48,1.520833,0.260680,0.000000,0.812500,48
2,457705,2025-05-11,0.274772,0.366160,0.399297,0.413562,0.327504,0.322416,0.327371,-0.021942,-0.021409,98,1.918367,0.351806,0.357143,0.020408,98
3,457759,2025-05-11,0.443777,0.286520,0.261328,0.282639,0.301951,0.308217,0.322703,0.005244,0.010843,55,1.654545,0.311287,0.127273,0.145455,55
4,467793,2025-05-11,0.322851,0.288303,0.337477,0.322596,0.322525,0.284948,0.282033,0.001919,0.007082,110,1.854545,0.321326,0.100000,0.054545,110


In [12]:
watchlist["pred_ridge"].describe()

count    339.000000
mean       0.319936
std        0.019307
min        0.223175
25%        0.308762
50%        0.322610
75%        0.332849
max        0.378673
Name: pred_ridge, dtype: float64

In [13]:
high_pred_threshold = watchlist["pred_ridge"].quantile(0.80)
low_pred_threshold = watchlist["pred_ridge"].quantile(0.20)

high_pred_threshold, low_pred_threshold

(np.float64(0.33491501303713495), np.float64(0.3037585281749986))

In [14]:
def assign_watchlist_signal(row):
    pred = row["pred_ridge"]
    season = row["pred_season_xwOBA"]
    last_7d = row["pred_last_7d_xwOBA"]
    last_14d = row["pred_last_14d_xwOBA"]

    avg_weakness = row.get("avg_matchup_weakness_score", np.nan)
    favorable_rate = row.get("favorable_matchup_rate", np.nan)
    unfavorable_rate = row.get("unfavorable_matchup_rate", np.nan)

    # Hot Candidate:
    # high model prediction and not matchup-risk-heavy
    if (
        pred >= high_pred_threshold
        and (pd.isna(avg_weakness) or avg_weakness <= 0.02)
    ):
        return "Hot Candidate"

    # Bounce-back:
    # model predicts better than recent/season context
    if (
        pred > season + 0.03
        and pred > last_14d + 0.03
        and (pd.isna(avg_weakness) or avg_weakness <= 0.02)
    ):
        return "Bounce-back Candidate"

    # Regression Risk:
    # model prediction lower than recent form or matchup context risky
    if (
        pred < last_7d - 0.04
        or pred < last_14d - 0.04
        or (pd.notna(unfavorable_rate) and unfavorable_rate >= 0.40)
    ):
        return "Regression Risk"

    return "Neutral"


watchlist["signal"] = watchlist.apply(assign_watchlist_signal, axis=1)

watchlist["signal"].value_counts()

signal
Regression Risk          165
Neutral                   93
Hot Candidate             67
Bounce-back Candidate     14
Name: count, dtype: int64

In [15]:
def build_reason(row):
    reasons = []

    if row["signal"] == "Hot Candidate":
        reasons.append("High Ridge-predicted future 7d xwOBA")
        if pd.notna(row.get("avg_matchup_weakness_score")):
            reasons.append("matchup profile is not strongly unfavorable")

    elif row["signal"] == "Bounce-back Candidate":
        reasons.append("model prediction is meaningfully higher than recent/season baseline")
        reasons.append("potential positive regression candidate")

    elif row["signal"] == "Regression Risk":
        reasons.append("model prediction is below recent-form baseline or matchup risk is elevated")

    else:
        reasons.append("no strong positive or negative signal")

    return "; ".join(reasons)


watchlist["reason"] = watchlist.apply(build_reason, axis=1)

In [16]:
watchlist_display_cols = [
    "batter",
    "prediction_date",
    "future_7d_xwOBA",
    "pred_ridge",
    "pred_season_xwOBA",
    "pred_last_7d_xwOBA",
    "pred_last_14d_xwOBA",
    "avg_matchup_weakness_score",
    "favorable_matchup_rate",
    "unfavorable_matchup_rate",
    "signal",
    "reason",
]

daily_watchlist = (
    watchlist[watchlist_display_cols]
    .sort_values(["signal", "pred_ridge"], ascending=[True, False])
)

daily_watchlist.head(30)

,batter,prediction_date,future_7d_xwOBA,pred_ridge,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,avg_matchup_weakness_score,favorable_matchup_rate,unfavorable_matchup_rate,signal,reason
217,671732,2025-05-11,0.262640,0.332884,0.292305,0.361335,0.286560,0.007430,0.045455,0.054545,Bounce-back Candidate,model prediction is meaningfully higher than r...
97,645277,2025-05-11,0.368409,0.325790,0.260828,0.190242,0.211834,0.012625,0.056911,0.219512,Bounce-back Candidate,model prediction is meaningfully higher than r...
39,593428,2025-05-11,0.482620,0.325705,0.287612,0.294519,0.286681,0.002308,0.000000,0.100000,Bounce-back Candidate,model prediction is meaningfully higher than r...
58,609280,2025-05-11,0.262164,0.325140,0.284737,0.256595,0.277790,0.018321,0.035714,0.321429,Bounce-back Candidate,model prediction is meaningfully higher than r...
146,664040,2025-05-11,0.494030,0.324992,0.289346,0.220034,0.289155,-0.017653,0.244898,0.000000,Bounce-back Candidate,model prediction is meaningfully higher than r...
275,682829,2025-05-11,0.352499,0.324817,0.276579,0.212718,0.243104,0.002222,0.000000,0.024793,Bounce-back Candidate,model prediction is meaningfully higher than r...
13,527038,2025-05-11,0.362720,0.321850,0.270282,0.307625,0.223807,0.018387,0.016000,0.200000,Bounce-back Candidate,model prediction is meaningfully higher than r...
321,696100,2025-05-11,0.336717,0.321172,0.270564,0.238062,0.226726,0.008848,0.008929,0.071429,Bounce-back Candidate,model prediction is meaningfully higher than r...
336,808975,2025-05-11,0.382010,0.318469,0.286789,0.286789,0.286789,-0.068367,0.774194,0.000000,Bounce-back Candidate,model prediction is meaningfully higher than r...
46,605137,2025-05-11,0.330009,0.309966,0.276719,0.158687,0.243130,-0.018173,0.343750,0.010417,Bounce-back Candidate,model prediction is meaningfully higher than r...


In [17]:
daily_watchlist[daily_watchlist["signal"] == "Hot Candidate"].head(20)

,batter,prediction_date,future_7d_xwOBA,pred_ridge,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,avg_matchup_weakness_score,favorable_matchup_rate,unfavorable_matchup_rate,signal,reason
123,660271,2025-05-11,0.505611,0.378673,0.473864,0.606946,0.568255,-0.156252,1.000000,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
160,665742,2025-05-11,0.404151,0.374260,0.431507,0.619680,0.607589,-0.095134,0.975610,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
244,677951,2025-05-11,0.422247,0.357877,0.380726,0.507017,0.413517,-0.046115,0.678899,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
164,665833,2025-05-11,0.006222,0.356575,0.392271,0.235011,0.348704,-0.088010,0.828283,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
34,592518,2025-05-11,0.390979,0.354617,0.415226,0.453340,0.450939,-0.080886,0.970297,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
66,621566,2025-05-11,0.344456,0.353864,0.367567,0.231633,0.313896,-0.052421,0.804878,0.008130,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
159,665489,2025-05-11,0.416929,0.353496,0.390161,0.364775,0.380279,-0.096734,1.000000,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
33,592450,2025-05-11,0.408845,0.352668,0.457196,0.463473,0.497545,-0.128312,1.000000,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
22,553993,2025-05-11,0.400062,0.351689,0.348208,0.466785,0.372744,-0.026497,0.418033,0.008197,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...
320,695578,2025-05-11,0.417522,0.351615,0.382081,0.382293,0.380550,-0.087737,0.825397,0.000000,Hot Candidate,High Ridge-predicted future 7d xwOBA; matchup ...


In [18]:
daily_watchlist[daily_watchlist["signal"] == "Regression Risk"].head(20)

,batter,prediction_date,future_7d_xwOBA,pred_ridge,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,avg_matchup_weakness_score,favorable_matchup_rate,unfavorable_matchup_rate,signal,reason
176,666969,2025-05-11,0.289452,0.338991,0.294289,0.365104,0.298103,0.048523,0.018692,0.691589,Regression Risk,model prediction is below recent-form baseline...
252,680574,2025-05-11,0.381596,0.334879,0.290553,0.330142,0.281577,0.030654,0.000000,0.425926,Regression Risk,model prediction is below recent-form baseline...
304,690993,2025-05-11,0.398053,0.334855,0.373441,0.528118,0.481401,-0.027324,0.402174,0.021739,Regression Risk,model prediction is below recent-form baseline...
332,702616,2025-05-11,0.379815,0.334702,0.345066,0.431429,0.391865,-0.013582,0.184466,0.009709,Regression Risk,model prediction is below recent-form baseline...
265,681546,2025-05-11,0.345970,0.334196,0.352592,0.352592,0.352592,-0.002095,0.388889,0.444444,Regression Risk,model prediction is below recent-form baseline...
152,664761,2025-05-11,0.368222,0.334144,0.294113,0.317715,0.377429,0.016228,0.000000,0.156522,Regression Risk,model prediction is below recent-form baseline...
178,667670,2025-05-11,0.343660,0.333820,0.364000,0.336550,0.378389,-0.054027,0.728814,0.008475,Regression Risk,model prediction is below recent-form baseline...
179,668227,2025-05-11,0.287084,0.333576,0.339124,0.444255,0.385260,0.001117,0.008475,0.025424,Regression Risk,model prediction is below recent-form baseline...
272,682663,2025-05-11,0.350049,0.333531,0.357434,0.386310,0.318566,-0.032802,0.594595,0.054054,Regression Risk,model prediction is below recent-form baseline...
329,701538,2025-05-11,0.251083,0.332930,0.462742,0.407318,0.407318,-0.048290,0.400000,0.000000,Regression Risk,model prediction is below recent-form baseline...


In [19]:
watchlist_path = PROJECT_ROOT / f"data/predictions/watchlist_demo_{demo_date.date()}.csv"

save_csv(daily_watchlist, watchlist_path)

print(watchlist_path)
print(watchlist_path.exists())

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/data/predictions/watchlist_demo_2025-05-11.csv
True


In [20]:
watchlist_summary = (
    daily_watchlist
    .groupby("signal")
    .agg(
        n_hitters=("batter", "count"),
        avg_pred_ridge=("pred_ridge", "mean"),
        avg_actual_future_7d_xwOBA=("future_7d_xwOBA", "mean"),
        avg_matchup_weakness_score=("avg_matchup_weakness_score", "mean"),
    )
    .reset_index()
    .sort_values("avg_pred_ridge", ascending=False)
)

watchlist_summary

,signal,n_hitters,avg_pred_ridge,avg_actual_future_7d_xwOBA,avg_matchup_weakness_score
1,Hot Candidate,67,0.344269,0.347265,-0.045653
2,Neutral,93,0.316813,0.317145,-0.001050
0,Bounce-back Candidate,14,0.312799,0.362746,-0.000966
3,Regression Risk,165,0.312421,0.283469,0.023629


In [21]:
watchlist_summary_path = PROJECT_ROOT / f"reports/watchlist_summary_demo_{demo_date.date()}.csv"

save_csv(watchlist_summary, watchlist_summary_path)

print(watchlist_summary_path.exists())

True
